# Stage 7 — KPI Design

This notebook validates the KPI framework against the star-schema tables. It calculates baseline values for the full period, 1997 and 1998, and checks the definitions that will be implemented in Power BI.

## Key assumptions

- Profit means gross product profit, not net profit.
- Sales rows are product-level sales lines, not orders.
- Same-store growth uses the 13 stores active in both 1997 and 1998.
- Return Quantity Rate is a directional product/store measure because returns cannot be matched to the original sales line.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path(os.getenv("MAVEN_MARKET_PROJECT_ROOT", Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.getenv("MAVEN_MARKET_OUTPUT_ROOT", PROJECT_ROOT)).resolve()

script_path = PROJECT_ROOT / "scripts" / "calculate_kpis.py"
if not script_path.exists():
    # When this notebook is executed from a separate incremental package.
    script_path = OUTPUT_ROOT / "scripts" / "calculate_kpis.py"

sys.path.insert(0, str(script_path.parent))
from calculate_kpis import load_model_tables, add_year_columns, build_snapshot, kpi_dictionary, validate

print(f"Project root: {PROJECT_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

Project root: /mnt/data/maven-market-retail-intelligence-phase-05-data-modeling
Output root: /mnt/data/maven-market-retail-intelligence-phase-07-kpi-only


## Load validated model tables

In [2]:
tables = load_model_tables(PROJECT_ROOT)
add_year_columns(tables)

{
    name: frame.shape
    for name, frame in tables.items()
}

{'sales': (269720, 17),
 'returns': (7087, 9),
 'date': (737, 15),
 'customer': (10281, 24),
 'product': (1560, 13),
 'store': (24, 17)}

## Calculate KPI baselines

In [3]:
snapshot = build_snapshot(tables)
snapshot

,metric_id,overall,1997,1998
0,total_revenue,1.764546e+06,565238.130000,1.199308e+06
1,total_product_cost,7.117277e+05,228112.650000,4.836150e+05
2,gross_product_profit,1.052819e+06,337125.480000,7.156933e+05
3,gross_margin_pct,5.966512e-01,0.596431,5.967551e-01
4,revenue_growth_pct,NaN,NaN,1.121775e+00
5,profit_growth_pct,NaN,NaN,1.122929e+00
6,same_store_revenue_growth_pct,NaN,NaN,8.400031e-02
7,same_store_profit_growth_pct,NaN,NaN,8.465542e-02
8,quantity_sold,8.334890e+05,266773.000000,5.667160e+05
9,sales_lines,2.697200e+05,86837.000000,1.828830e+05


## Executive scorecard values

In [4]:
executive_ids = [
    "total_revenue",
    "gross_product_profit",
    "gross_margin_pct",
    "revenue_growth_pct",
    "same_store_revenue_growth_pct",
    "quantity_sold",
    "active_customers",
    "return_quantity_rate_pct",
]

snapshot.loc[snapshot["metric_id"].isin(executive_ids)].reset_index(drop=True)

,metric_id,overall,1997,1998
0,total_revenue,1.764546e+06,565238.130000,1.199308e+06
1,gross_product_profit,1.052819e+06,337125.480000,7.156933e+05
2,gross_margin_pct,5.966512e-01,0.596431,5.967551e-01
3,revenue_growth_pct,NaN,NaN,1.121775e+00
4,same_store_revenue_growth_pct,NaN,NaN,8.400031e-02
5,quantity_sold,8.334890e+05,266773.000000,5.667160e+05
6,active_customers,8.842000e+03,5581.000000,8.060000e+03
7,return_quantity_rate_pct,9.944942e-03,0.009889,9.971485e-03


## KPI dictionary coverage

In [5]:
dictionary = kpi_dictionary()
dictionary.groupby("group").size().rename("kpi_count").to_frame()

,kpi_count
group,
Executive,8
Operational,10


## Validation

In [6]:
validation = validate(snapshot)
validation

{'status': 'passed',
 'checks': {'overall_revenue_matches_model': True,
  'overall_profit_matches_model': True,
  'overall_quantity_matches_model': True,
  'reported_growth_exceeds_same_store_growth': True,
  'return_rate_is_between_zero_and_one': True,
  'gross_margin_is_between_zero_and_one': True,
  'no_metric_definition_duplicates': True},
 'notes': ['No target values were created because the dataset contains no budget, plan or management threshold.',
  'Order count and average order value are excluded because the source has no order identifier.',
  'Return Quantity Rate must not be analysed by customer because FactReturns has no customer key.']}

## Takeaways

- Reported revenue growth must be read beside same-store revenue growth.
- Gross margin is a required guardrail because total revenue expanded mainly through a larger store base.
- Order-based KPIs are excluded because the source has no order ID.
- No traffic-light targets are added until the business provides approved target values.